# Regresión Lineal - Implementación desde Cero

Bienvenido al primer notebook de algoritmos supervisados. La Regresión Lineal es el algoritmo más fundamental en Machine Learning y la base para entender algoritmos más complejos.

Al finalizar este notebook, serás capaz de:

* Comprender la matemática completa detrás de la regresión lineal
* Implementar el algoritmo desde cero usando solo NumPy
* Entender y aplicar Gradient Descent para optimización
* Trabajar con regresión univariada y multivariada
* Implementar normalización de features para mejor convergencia
* Evaluar modelos usando métricas apropiadas (MSE, R²)
* Diagnosticar y prevenir overfitting

**¿Por qué empezar con Regresión Lineal?**

1. **Fundamento matemático**: Introduce conceptos clave (loss function, gradient descent)
2. **Simplicidad**: Fácil de entender e implementar
3. **Base para otros algoritmos**: Regresión logística, redes neuronales, etc.
4. **Aplicable**: Útil en muchos problemas reales (precios, ventas, demanda)
5. **Interpretable**: Los pesos tienen significado claro

## Nota Importante sobre los Ejercicios

Antes de comenzar con los ejercicios, ten en cuenta lo siguiente:

1. NO agregues declaraciones `print` adicionales en las funciones graduadas
2. NO agregues celdas de código adicionales entre los ejercicios
3. NO cambies los parámetros de las funciones
4. Implementa usando NumPy (operaciones vectorizadas, sin bucles cuando sea posible)
5. NO cambies el código de las pruebas automáticas

Si experimentas errores al ejecutar las pruebas, primero verifica estos puntos antes de buscar ayuda.

<a name='1'></a>
## Tabla de Contenidos
- [1 - Paquetes](#1)
- [2 - Teoría de Regresión Lineal](#2)
- [3 - Implementación desde Cero](#3)
  - [Ejercicio 1](#ex01)
- [4 - Regresión Univariada](#4)
- [5 - Regresión Multivariada](#5)
- [6 - Normalización de Features](#6)
  - [Ejercicio 2](#ex02)
- [7 - Referencias](#7)

<a name='1'></a>
## 1 - Paquetes

Ejecuta la siguiente celda para importar los paquetes que usarás en este notebook:

* **NumPy**: Operaciones numéricas y álgebra lineal
* **Matplotlib**: Visualización de resultados y gráficos
* **Testing utilities**: Verificación automática de ejercicios

In [ ]:
# ==========================================
# CONFIGURACIÓN DEL ENTORNO
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Configurar matplotlib inline
%matplotlib inline

# Agregar el directorio raíz al path
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Importar utilidades de testing
from utils.testing_utils import print_success, print_error, print_info
from tests.supervisados.test_01_regresion_linear import (
    test_ejercicio_1_predict,
    test_ejercicio_2_gradient_descent
)

print("✅ Paquetes importados correctamente")
print(f"📦 NumPy version: {np.__version__}")

<a name='2'></a>
## 2 - Teoría de Regresión Lineal

La regresión lineal modela la relación entre una o más variables independientes (features) y una variable dependiente continua (target).

### 2.1 - El Modelo Lineal

Para un problema con $n$ features, el modelo es:

$$\hat{y} = w_0 + w_1x_1 + w_2x_2 + ... + w_nx_n$$

En notación vectorial (más compacta):

$$\hat{y} = \mathbf{w}^T \mathbf{x} + b$$

donde:
- $\hat{y}$ = valor predicho (scalar)
- $\mathbf{w} = [w_1, w_2, ..., w_n]^T$ = vector de pesos (weights)
- $\mathbf{x} = [x_1, x_2, ..., x_n]^T$ = vector de features
- $b = w_0$ = sesgo (bias o intercept)

### 2.2 - Función de Costo (Loss Function)

Necesitamos medir qué tan lejos están nuestras predicciones de los valores reales. Usamos **Mean Squared Error (MSE)**:

$$J(\mathbf{w}, b) = \frac{1}{2m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})^2$$

donde:
- $m$ = número de muestras de entrenamiento
- $\hat{y}^{(i)}$ = predicción para la muestra $i$
- $y^{(i)}$ = valor real para la muestra $i$
- El factor $\frac{1}{2}$ simplifica las derivadas

**Objetivo:** Minimizar $J(\mathbf{w}, b)$ encontrando los mejores valores de $\mathbf{w}$ y $b$.

### 2.3 - Gradient Descent

Es un algoritmo iterativo para encontrar el mínimo de la función de costo:

$$\mathbf{w} := \mathbf{w} - \alpha \frac{\partial J}{\partial \mathbf{w}}$$

$$b := b - \alpha \frac{\partial J}{\partial b}$$

donde:
- $\alpha$ = learning rate (tasa de aprendizaje)
- $\frac{\partial J}{\partial \mathbf{w}}$ = gradiente respecto a los pesos
- $\frac{\partial J}{\partial b}$ = gradiente respecto al sesgo

### 2.4 - Cálculo de Gradientes

Las derivadas parciales son:

$$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{m} \mathbf{X}^T (\hat{\mathbf{y}} - \mathbf{y})$$

$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})$$

donde:
- $\mathbf{X}$ = matriz de features de forma $(m, n)$
- $\hat{\mathbf{y}}$ = vector de predicciones de forma $(m,)$
- $\mathbf{y}$ = vector de valores reales de forma $(m,)$

<a name='3'></a>
## 3 - Implementación desde Cero

Vamos a implementar una clase completa de Regresión Lineal que utiliza **Gradient Descent** para encontrar los parámetros óptimos.

### 3.1 - Estructura de la Clase

Nuestra clase tendrá los siguientes métodos:

* `__init__(learning_rate, n_iterations)`: Inicializa hiperparámetros
* `fit(X, y)`: Entrena el modelo usando Gradient Descent
* `predict(X)`: Hace predicciones con el modelo entrenado
* `score(X, y)`: Calcula el coeficiente de determinación $R^2$

### 3.2 - Código de Implementación

Ejecuta la siguiente celda para ver la implementación completa:

In [ ]:
class RegresionLinear:
    """
    Regresión Linear implementada desde cero con Gradient Descent.
    
    Parámetros:
    -----------
    learning_rate : float
        Tasa de aprendizaje (alpha)
    n_iterations : int
        Número de iteraciones para gradient descent
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def fit(self, X, y):
        """
        Entrena el modelo usando Gradient Descent.
        
        Parámetros:
        -----------
        X : array-like, shape (n_samples, n_features)
            Features de entrenamiento
        y : array-like, shape (n_samples,)
            Target de entrenamiento
        """
        # Convertir a numpy arrays
        X = np.array(X)
        y = np.array(y)
        
        # Si X es 1D, convertir a 2D
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        
        # Número de muestras y features
        n_samples, n_features = X.shape
        
        # Inicializar parámetros
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Gradient Descent
        for i in range(self.n_iterations):
            # Predicción
            y_pred = self.predict(X)
            
            # Calcular gradientes
            dw = (1/n_samples) * np.dot(X.T, (y_pred - y))
            db = (1/n_samples) * np.sum(y_pred - y)
            
            # Actualizar parámetros
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            # Guardar loss
            loss = self._mse(y, y_pred)
            self.loss_history.append(loss)
            
            # Imprimir progreso cada 100 iteraciones
            if (i + 1) % 100 == 0:
                print(f"Iteración {i+1}/{self.n_iterations}, Loss: {loss:.4f}")
        
        return self
    
    def predict(self, X):
        """
        Hace predicciones.
        
        Parámetros:
        -----------
        X : array-like, shape (n_samples, n_features)
            Features para predecir
        
        Returns:
        --------
        array, shape (n_samples,)
            Predicciones
        """
        X = np.array(X)
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        
        return np.dot(X, self.weights) + self.bias
    
    def _mse(self, y_true, y_pred):
        """Calcula Mean Squared Error"""
        return np.mean((y_true - y_pred) ** 2)
    
    def score(self, X, y):
        """
        Calcula el R² score.
        
        Returns:
        --------
        float
            R² score
        """
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

print("✅ Clase RegresionLinear definida correctamente")

<a name='4'></a>
## 4 - Regresión Univariada

Empezamos con un ejemplo simple: predecir $y$ a partir de una sola variable $x$.

### 4.1 - Generar Datos Sintéticos

Vamos a crear datos con la relación: $y = 4 + 3x + \text{ruido}$

In [ ]:
# Generar datos sintéticos
np.random.seed(42)
X = 2 * np.random.rand(100)
y = 4 + 3 * X + np.random.randn(100)  # y = 4 + 3x + ruido

print(f"Datos generados: {len(X)} muestras")
print(f"Relación real: y = 4 + 3x + ruido")

# Visualizar datos
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Datos de Entrenamiento')
plt.grid(True, alpha=0.3)
plt.show()

### 4.2 - Entrenar el Modelo

Ahora entrenamos nuestro modelo y comparamos los parámetros aprendidos con los reales:

In [ ]:
# Entrenar el modelo
modelo = RegresionLinear(learning_rate=0.1, n_iterations=1000)
modelo.fit(X, y)

print(f"\nParámetros aprendidos:")
print(f"Peso (w): {modelo.weights[0]:.4f}")
print(f"Bias (b): {modelo.bias:.4f}")
print(f"\nEcuación aprendida: y = {modelo.bias:.4f} + {modelo.weights[0]:.4f}x")
print(f"Ecuación real:      y = 4.0000 + 3.0000x")

### 4.3 - Visualizar Resultados

Vamos a graficar los datos reales junto con la línea de predicción del modelo:

In [ ]:
# Visualizar resultados
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, label='Datos reales')

# Línea de predicción
X_plot = np.linspace(X.min(), X.max(), 100)
y_plot = modelo.predict(X_plot)
plt.plot(X_plot, y_plot, color='red', linewidth=2, label='Predicción')

plt.xlabel('X')
plt.ylabel('y')
plt.title('Regresión Linear - Ajuste del Modelo')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 4.4 - Curva de Aprendizaje

La curva de aprendizaje muestra cómo el loss disminuye durante el entrenamiento:

In [ ]:
# Visualizar curva de aprendizaje (cómo baja el loss)
plt.figure(figsize=(10, 6))
plt.plot(modelo.loss_history, linewidth=2)
plt.xlabel('Iteración')
plt.ylabel('Loss (MSE)')
plt.title('Curva de Aprendizaje - Convergencia de Gradient Descent')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Loss inicial: {modelo.loss_history[0]:.4f}")
print(f"Loss final: {modelo.loss_history[-1]:.4f}")
print(f"Reducción: {(1 - modelo.loss_history[-1]/modelo.loss_history[0]) * 100:.2f}%")

<a name='5'></a>
## 5 - Regresión Multivariada

Ahora trabajaremos con múltiples features (variables independientes).

### 5.1 - Generar Datos con Múltiples Features

Creamos un dataset con 3 features y la relación: $y = 5 + 2x_1 - 3x_2 + 4x_3 + \text{ruido}$

In [ ]:
# Generar datos con 3 features
np.random.seed(42)
n_samples = 200
X_multi = np.random.rand(n_samples, 3)

# Relación real: y = 5 + 2*x1 - 3*x2 + 4*x3 + ruido
y_multi = 5 + 2*X_multi[:, 0] - 3*X_multi[:, 1] + 4*X_multi[:, 2] + np.random.randn(n_samples) * 0.5

print(f"Datos generados: {n_samples} muestras, 3 features")
print(f"Relación real: y = 5 + 2*x1 - 3*x2 + 4*x3 + ruido")

### 5.2 - División Train/Test

Dividimos los datos para evaluar la capacidad de generalización del modelo:

In [ ]:
# Train/Test split manual
def train_test_split(X, y, test_size=0.2, random_state=None):
    if random_state:
        np.random.seed(random_state)
    
    n = len(X)
    indices = np.random.permutation(n)
    test_size_n = int(n * test_size)
    
    test_idx = indices[:test_size_n]
    train_idx = indices[test_size_n:]
    
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

print(f"Train set: {len(X_train)} muestras")
print(f"Test set: {len(X_test)} muestras")

### 5.3 - Entrenar y Evaluar el Modelo

Entrenamos el modelo y verificamos que aprende los parámetros correctos:

In [ ]:
# Entrenar modelo
modelo_multi = RegresionLinear(learning_rate=0.1, n_iterations=1000)
modelo_multi.fit(X_train, y_train)

print(f"\nParámetros aprendidos:")
print(f"Bias: {modelo_multi.bias:.4f}")
print(f"Pesos: {modelo_multi.weights}")
print(f"\nParámetros reales:")
print(f"Bias: 5.0000")
print(f"Pesos: [2.0000, -3.0000, 4.0000]")

### 5.4 - Evaluación con $R^2$ Score

El coeficiente de determinación $R^2$ mide qué tan bien el modelo ajusta los datos:

$$R^2 = 1 - \frac{\sum_{i=1}^{m} (y^{(i)} - \hat{y}^{(i)})^2}{\sum_{i=1}^{m} (y^{(i)} - \bar{y})^2}$$

donde $\bar{y}$ es la media de los valores reales. Valores cercanos a 1 indican buen ajuste.

In [ ]:
# Evaluar en train y test
train_score = modelo_multi.score(X_train, y_train)
test_score = modelo_multi.score(X_test, y_test)

print(f"R² en Training: {train_score:.4f}")
print(f"R² en Test: {test_score:.4f}")

if abs(train_score - test_score) < 0.1:
    print("\n✅ El modelo generaliza bien (no hay overfitting)")
else:
    print("\n⚠️  Posible overfitting")

<a name='6'></a>
## 6 - Normalización de Features

Cuando las features tienen escalas muy diferentes, normalizar ayuda a que gradient descent converja más rápido y de manera más estable.

### 6.1 - El Problema de Escalas Diferentes

Consideremos features con rangos muy distintos:
- Feature 1: valores entre 0 y 1000
- Feature 2: valores entre 0 y 1

Sin normalización, los gradientes tendrán magnitudes muy diferentes, lo que dificulta encontrar un learning rate adecuado.

### 6.2 - Z-Score Normalization

La normalización Z-score transforma cada feature para tener media 0 y desviación estándar 1:

$$x_{\text{norm}} = \frac{x - \mu}{\sigma}$$

donde $\mu$ es la media y $\sigma$ es la desviación estándar.

### 6.3 - Ejemplo Práctico

Vamos a comparar el entrenamiento con y sin normalización:

<a name='7'></a>
## 7 - Referencias

### Papers Fundamentales

1. **Legendre, A. M.** (1805). *Nouvelles méthodes pour la détermination des orbites des comètes*. 
   - Primera publicación del método de mínimos cuadrados

2. **Gauss, C. F.** (1809). *Theoria motus corporum coelestium*. 
   - Desarrolló la teoría de mínimos cuadrados de forma independiente

3. **Rumelhart, D. E., Hinton, G. E., & Williams, R. J.** (1986). "Learning representations by back-propagating errors." *Nature*, 323(6088), 533-536.
   - Generalización de gradient descent a redes neuronales

### Recursos Adicionales

4. **Bishop, C. M.** (2006). *Pattern Recognition and Machine Learning*. Springer. Chapter 3: Linear Models for Regression.
   - Tratamiento matemático riguroso de regresión lineal

5. **Murphy, K. P.** (2012). *Machine Learning: A Probabilistic Perspective*. MIT Press. Chapter 7: Linear Regression.
   - Perspectiva probabilística de regresión lineal

6. **James, G., Witten, D., Hastie, T., & Tibshirani, R.** (2013). *An Introduction to Statistical Learning*. Springer. Chapter 3: Linear Regression.
   - Introducción accesible con ejemplos en R

### Recursos Online

7. **Andrew Ng's Machine Learning Course** - Coursera
   - https://www.coursera.org/learn/machine-learning
   - Excelente introducción práctica a regresión lineal

8. **Scikit-learn Documentation: Linear Models**
   - https://scikit-learn.org/stable/modules/linear_model.html
   - Implementaciones optimizadas y documentación técnica

9. **3Blue1Brown: Gradient Descent**
   - https://www.youtube.com/watch?v=IHZwWFHWa-w
   - Visualización intuitiva de gradient descent

### Extensiones y Variantes

10. **Ridge Regression** (L2 Regularization): Añade penalización $\lambda \|\mathbf{w}\|^2$ para prevenir overfitting
11. **Lasso Regression** (L1 Regularization): Añade penalización $\lambda \|\mathbf{w}\|_1$ para feature selection
12. **Elastic Net**: Combina L1 y L2 regularization

## 📘 Resumen y Aplicaciones en ML

<div style="background-color: #e7f3fe; padding: 20px; border-left: 6px solid #2196F3; margin: 20px 0;">

**Conceptos Clave Aprendidos:**

1. **Modelo Lineal**: $\hat{y} = \mathbf{w}^T \mathbf{x} + b$ es la base de muchos algoritmos de ML
2. **Función de Costo**: MSE mide el error cuadrático promedio entre predicciones y valores reales
3. **Gradient Descent**: Algoritmo de optimización iterativo que minimiza la función de costo
4. **Normalización**: Esencial para features con escalas diferentes, mejora la convergencia
5. **Evaluación**: $R^2$ score mide la bondad de ajuste del modelo

**Relevancia en Machine Learning:**

- **Fundamento**: Los conceptos de loss function y gradient descent se usan en TODOS los algoritmos de ML
- **Redes Neuronales**: Son esencialmente regresiones lineales apiladas con funciones de activación
- **Regresión Logística**: Usa el mismo framework pero con función sigmoide para clasificación
- **Regularización**: L1 y L2 se agregan a la función de costo para prevenir overfitting
- **Aplicaciones Reales**: Predicción de precios, demanda, ventas, temperatura, etc.

**¿Por qué es importante dominar esto?**

Entender regresión lineal en profundidad te permite:
- Comprender cómo funcionan internamente los modelos de ML
- Debuggear problemas de convergencia y overfitting
- Implementar tus propias variantes y mejoras
- Pasar a algoritmos más complejos con bases sólidas

</div>

In [ ]:
# GRADED FUNCTION: compute_gradients

def compute_gradients(X, y, w, b):
    """
    Calcula los gradientes de la función de costo respecto a w y b.
    
    Parámetros
    ----------
    X : ndarray
        Matriz de features de forma (m, n)
    y : ndarray
        Vector de targets de forma (m,)
    w : ndarray
        Vector de pesos actual de forma (n,)
    b : float
        Sesgo actual
    
    Retorna
    -------
    tuple
        (dw, db) donde:
        - dw: gradiente respecto a w, forma (n,)
        - db: gradiente respecto a b, escalar
    
    Ejemplo
    -------
    >>> X = np.array([[1, 2], [3, 4]])
    >>> y = np.array([3, 7])
    >>> w = np.array([0.5, 0.5])
    >>> b = 0.0
    >>> dw, db = compute_gradients(X, y, w, b)
    """
    
    m = X.shape[0]  # número de muestras
    
    ### YOUR CODE STARTS HERE ###
    # Paso 1: Calcular predicciones
    y_pred = None
    
    # Paso 2: Calcular gradiente para w
    dw = None
    
    # Paso 3: Calcular gradiente para b
    db = None
    ### YOUR CODE ENDS HERE ###
    
    return dw, db

# Prueba tu implementación
X_test = np.array([[1, 2], [3, 4], [5, 6]])
y_test = np.array([5, 11, 17])
w_test = np.array([1.0, 1.0])
b_test = 1.0

dw, db = compute_gradients(X_test, y_test, w_test, b_test)
print(f"Gradiente dw: {dw}")
print(f"Gradiente db: {db}")

# Verificar con test automático
verificar_gradients = test_ejercicio_2_gradient_descent()
verificar_gradients(compute_gradients)

<a name='ex02'></a>
### Ejercicio 2: Implementar Paso de Gradient Descent

Implementa una función que ejecute un paso de gradient descent y retorne los gradientes calculados.

**Instrucciones:**
- Calcula los gradientes usando las fórmulas vectorizadas
- $\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{m} \mathbf{X}^T (\hat{\mathbf{y}} - \mathbf{y})$
- $\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})$
- Retorna una tupla (dw, db)

In [ ]:
# GRADED FUNCTION: linear_predict

def linear_predict(X, w, b):
    """
    Calcula predicciones usando el modelo lineal.
    
    Parámetros
    ----------
    X : ndarray
        Matriz de features de forma (m, n) donde m = muestras, n = features
    w : ndarray
        Vector de pesos de forma (n,)
    b : float
        Sesgo (bias)
    
    Retorna
    -------
    ndarray
        Vector de predicciones de forma (m,)
    
    Ejemplo
    -------
    >>> X = np.array([[1, 2], [3, 4], [5, 6]])
    >>> w = np.array([0.5, 0.3])
    >>> b = 1.0
    >>> linear_predict(X, w, b)
    array([2.1, 3.7, 5.3])
    """
    
    ### YOUR CODE STARTS HERE ###
    y_pred = None
    ### YOUR CODE ENDS HERE ###
    
    return y_pred

# Prueba tu implementación
X_test = np.array([[1, 2], [3, 4], [5, 6]])
w_test = np.array([0.5, 0.3])
b_test = 1.0

resultado = linear_predict(X_test, w_test, b_test)
print(f"Predicciones: {resultado}")

# Verificar con test automático
verificar_predict = test_ejercicio_1_predict()
verificar_predict(linear_predict)

<a name='ex01'></a>
### Ejercicio 1: Implementar Función de Predicción

Implementa una función que realice predicciones usando el modelo lineal $\hat{y} = \mathbf{w}^T \mathbf{x} + b$.

**Instrucciones:**
- Usa operaciones vectorizadas de NumPy (np.dot)
- No uses bucles
- Retorna un array 1D con las predicciones

In [ ]:
# Comparar convergencia con y sin normalización
modelo_sin_norm = RegresionLinear(learning_rate=0.0001, n_iterations=1000)
modelo_sin_norm.fit(X_sin_normalizar, y_escala)

modelo_con_norm = RegresionLinear(learning_rate=0.01, n_iterations=1000)
modelo_con_norm.fit(X_normalizado, y_escala)

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(modelo_sin_norm.loss_history)
axes[0].set_title('SIN Normalización\n(learning_rate=0.0001)')
axes[0].set_xlabel('Iteración')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(modelo_con_norm.loss_history)
axes[1].set_title('CON Normalización\n(learning_rate=0.01)')
axes[1].set_xlabel('Iteración')
axes[1].set_ylabel('Loss')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Loss final SIN normalización: {modelo_sin_norm.loss_history[-1]:.4f}")
print(f"Loss final CON normalización: {modelo_con_norm.loss_history[-1]:.4f}")

In [ ]:
def normalizar(X):
    """Normalización: (X - media) / std"""
    return (X - np.mean(X, axis=0)) / np.std(X, axis=0)

X_normalizado = normalizar(X_sin_normalizar)

print("Estadísticas de features NORMALIZADAS:")
print(f"Feature 1 - media: {X_normalizado[:, 0].mean():.2f}, std: {X_normalizado[:, 0].std():.2f}")
print(f"Feature 2 - media: {X_normalizado[:, 1].mean():.2f}, std: {X_normalizado[:, 1].std():.2f}")

In [ ]:
# Datos con escalas muy diferentes
np.random.seed(42)
X_sin_normalizar = np.random.rand(100, 2)
X_sin_normalizar[:, 0] *= 1000  # Primera feature: 0-1000
X_sin_normalizar[:, 1] *= 1     # Segunda feature: 0-1

y_escala = 5 + 2*X_sin_normalizar[:, 0] + 3*X_sin_normalizar[:, 1] + np.random.randn(100)*10

print("Estadísticas de features SIN normalizar:")
print(f"Feature 1 - min: {X_sin_normalizar[:, 0].min():.2f}, max: {X_sin_normalizar[:, 0].max():.2f}")
print(f"Feature 2 - min: {X_sin_normalizar[:, 1].min():.2f}, max: {X_sin_normalizar[:, 1].max():.2f}")